In [1]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [2]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [3]:
compact = True

In [4]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading fixed.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4896996752,<NA>,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5130,0.015390,0,"[8, 4, 0, 0, 0]"
6158719728,4896996752,"[41, 30, 42]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6077,0.024308,1,"[4, 8, 3, 2, 0]"
6158687296,6158719728,"[41, 30, 42, 6]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.5770,0.046160,1,"[4, 8, 6, 3, 0]"
6158493008,6158687296,"[41, 30, 42, 6, 28]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5421,0.086736,2,"[4, 2, 8, 0, 0]"
6158678432,6158493008,"[41, 30, 42, 6, 28]","[84, 78]",0,"[0, 6]","[16, 22]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5421,0.102999,2,"[4, 2, 8, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6127529072,"[8, 9, 36]","[152, 40]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6240,0.024960,1,"[8, 10, 9, 4, 0]"
6431104400,6431248928,"[8, 9, 36, 6]","[148, 36]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5459,0.043672,1,"[8, 10, 9, 6, 0]"
6127537840,6431104400,"[8, 9, 36, 6]","[144, 26]",0,"[4, 10]","[12, 18]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5459,0.081885,1,"[8, 10, 9, 6, 0]"


In [5]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

In [6]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [7]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name,n_players
4896996752,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.5130,0.015390,preflop,Tord,2
6158719728,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.6077,0.024308,flop,Tord,2
6158687296,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,4,0,0,0,2,0,0,0,...,0,0,raise,8,1,0.5770,0.046160,turn,Tord,2
6158493008,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,4,8,0,0,2,0,0,0,...,0,0,check,0,1,0.5421,0.086736,river,Tord,2
6158678432,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,4,8,0,0,2,0,0,0,...,0,0,call,6,1,0.5421,0.102999,river,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.6240,0.024960,flop,Tord,2
6431104400,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,4,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.5459,0.043672,turn,Tord,2
6127537840,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,4,4,0,0,2,0,0,0,...,0,0,call,6,1,0.5459,0.081885,turn,Tord,2
6431250896,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,4,4,0,0,2,0,6,0,...,1,0,check,0,0,0.9412,0.169416,river,Tord,2


In [8]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [10]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [11]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name,n_players
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2
6158719728,0,0,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,flop,Tord,2
6158687296,0,4,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,8,turn,Tord,2
6158493008,0,4,8,0,0,2,0,0,0,0,...,0,1,0,0,0,check,0,river,Tord,2
6158678432,0,4,8,0,0,2,0,0,0,0,...,0,1,0,0,0,call,6,river,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,0,0,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,flop,Tord,2
6431104400,0,4,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,turn,Tord,2
6127537840,0,4,4,0,0,2,0,0,0,0,...,0,1,0,0,0,call,6,turn,Tord,2
6431250896,0,4,4,0,0,2,0,6,0,0,...,0,1,0,1,0,check,0,river,Tord,2


In [12]:
y

4896996752    0
6158719728    1
6158687296    1
6158493008    1
6158678432    1
             ..
6431248928    1
6431104400    1
6127537840    1
6431250896    0
6431104448    0
Name: excess_rank, Length: 3741, dtype: int64

In [13]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [14]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (2988, 35)
Test shape: (753, 35)


In [15]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

In [ ]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7327001356852103
Mean goodness:  0.6771498893596768


,0,1,2,3,4,5,true,pred,correct,goodness
6062049552,0.965425,0.033389,0.000363,0.000513,0.000176,0.000135,0,0,True,0.965425
6152325536,0.915596,0.078835,0.003110,0.001195,0.000492,0.000771,0,0,True,0.915596
4673137536,0.995689,0.004213,0.000033,0.000035,0.000018,0.000013,0,0,True,0.995689
6151996464,0.995689,0.004213,0.000033,0.000035,0.000018,0.000013,0,0,True,0.995689
6073122528,0.986673,0.013040,0.000058,0.000081,0.000092,0.000057,0,0,True,0.986673
...,...,...,...,...,...,...,...,...,...,...
6127527200,0.979713,0.018769,0.000128,0.000495,0.000279,0.000615,0,0,True,0.979713
6127534432,0.753550,0.231223,0.006950,0.002530,0.003354,0.002393,1,0,False,0.231223
6127526816,0.672332,0.321390,0.000018,0.002520,0.003612,0.000128,1,0,False,0.321390
6433956432,0.655075,0.314491,0.000016,0.010173,0.019744,0.000500,1,0,False,0.314491


In [ ]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

197 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
6075692560,0.778443,0.209346,0.004736,2.397511e-03,0.002093,0.002985,1,0,False,2.093462e-01
6075656272,0.816228,0.175355,0.005444,1.104669e-03,0.001315,0.000553,1,0,False,1.753551e-01
6075693328,0.736971,0.260378,0.000007,1.149630e-03,0.001477,0.000017,1,0,False,2.603785e-01
6075704896,0.533460,0.466092,0.000036,2.188798e-09,0.000244,0.000168,1,0,False,4.660920e-01
6075525440,0.003839,0.994599,0.000643,1.206754e-08,0.000685,0.000235,3,1,False,1.206754e-08
...,...,...,...,...,...,...,...,...,...,...
6127435936,0.104954,0.727223,0.117301,1.475933e-02,0.014034,0.021729,2,1,False,1.173012e-01
6127534432,0.753550,0.231223,0.006950,2.529945e-03,0.003354,0.002393,1,0,False,2.312234e-01
6127526816,0.672332,0.321390,0.000018,2.519963e-03,0.003612,0.000128,1,0,False,3.213900e-01
6433956432,0.655075,0.314491,0.000016,1.017260e-02,0.019744,0.000500,1,0,False,3.144915e-01


### Attempt to infer card probabilities from rank and table

In [ ]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank, CheatSheet

In [ ]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
6062049552,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6152325536,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4673137536,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6151996464,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6073122528,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6127527200,0,0,0,0,1,1,0,0,1,0,...,1,0,0,0,1,1,1,0,0,0
6127534432,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6127526816,1,1,1,1,1,2,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
6433956432,2,2,2,2,2,5,2,2,2,2,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
6062049552,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,...,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008,0.0008
6152325536,0.001134,0.000098,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134,0.000098,...,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134,0.001134
4673137536,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,...,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801
6151996464,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,...,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801
6073122528,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,...,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6127527200,0.001197,0.001197,0.001197,0.001197,0.000023,0.000023,0.001197,0.001197,0.000023,0.001197,...,0.000023,0.001197,0.001197,0.001197,0.000023,0.000023,0.000023,0.001197,0.001197,0.001197
6127534432,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,...,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786,0.000786
6127526816,0.000452,0.000452,0.000452,0.000452,0.000452,0.0,0.000452,0.000452,0.000452,0.000452,...,0.000946,0.000946,0.000946,0.000946,0.000946,0.000946,0.000946,0.000946,0.000946,0.000946
6433956432,0.0,0.0,0.0,0.0,0.0,0.000001,0.0,0.0,0.0,0.0,...,0.000955,0.000955,0.000955,0.000955,0.000955,0.000955,0.000955,0.000955,0.000955,0.000955


In [ ]:
# Validate that rows sum to 1
hand_probs_df.sum(axis=1)

6062049552    1.0
6152325536    1.0
4673137536    1.0
6151996464    1.0
6073122528    1.0
             ... 
6127527200    1.0
6127534432    1.0
6127526816    1.0
6433956432    1.0
6431104352    1.0
Length: 737, dtype: object

In [ ]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
min_probs = hand_probs_df[mask].min(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
min_prob_hands = hand_probs_df[mask].idxmin(axis=1)
mean_probs = hand_probs_df[mask].mean(axis=1)
sample_prob_df = pd.DataFrame(
    {
        "max_prob": max_probs,
        "max_prob_hand": max_prob_hands,
        "min_prob": min_probs,
        "min_prob_hand": min_prob_hands,
        "mean_prob": mean_probs,
    }
)
sample_prob_df["max_prob_hand"] = sample_prob_df["max_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["min_prob_hand"] = sample_prob_df["min_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["table"] = raw_df.loc[sample_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
sample_prob_df["predicted_excess_rank"] = prob_df.loc[sample_prob_df.index, "pred"]
sample_prob_df["pred_rank"] = (
    sample_prob_df["predicted_excess_rank"]
    + table_ranks_df.loc[sample_prob_df.index, "table_rank"]
)
sample_prob_df["pred_rank_str"] = sample_prob_df["pred_rank"].apply(
    lambda x: HandRank(x, []).get_rank_name()
)
sample_prob_df = sample_prob_df.drop(["predicted_excess_rank"], axis=1)
sample_prob_df

,max_prob,max_prob_hand,min_prob,min_prob_hand,mean_prob,table,pred_rank,pred_rank_str
6162604912,0.001508,♥ 6 ♥ 7,0.000027,♥ 6 ♣ 6,0.000754,♥ Q ♦ 6 ♦ J ♠ 7 ♠ 10,2,Two Pairs
6269893024,0.005208,♥ 9 ♥ J,0.0,♥ 9 ♣ 9,0.000754,♦ 9 ♦ J ♦ A,2,Two Pairs
6165563904,0.014645,♥ 9 ♥ J,0.0,♥ 9 ♣ 9,0.000754,♦ 9 ♦ J ♦ A,2,Two Pairs
6165442032,0.005163,♥ 9 ♥ J,0.0,♥ 9 ♣ 9,0.000754,♦ 9 ♦ J ♦ A,2,Two Pairs
6165562320,0.003795,♥ 9 ♥ J,0.0,♥ 9 ♣ 9,0.000754,♥ 8 ♦ 9 ♦ J ♦ A,2,Two Pairs
6165583184,0.002464,♥ 4 ♥ 9,0.0,♥ 4 ♦ 4,0.000754,♥ 8 ♦ 9 ♦ J ♦ A ♣ 4,2,Two Pairs
6097455760,0.010136,♥ 9 ♠ 9,0.000009,♥ 2 ♥ 3,0.000754,♦ 6 ♦ 9 ♦ J ♣ 9,3,Three of a Kind
6199120752,0.011667,♥ 9 ♠ 9,0.000007,♥ 2 ♥ 3,0.000754,♦ 6 ♦ 9 ♦ J ♣ 9,3,Three of a Kind
6100621776,0.011832,♥ 9 ♠ 9,0.0,♥ 2 ♥ 3,0.000754,♦ 6 ♦ 9 ♦ J ♦ K ♣ 9,3,Three of a Kind


In [ ]:
# Deepdive into a specific row
state_id = "6165563904"
row = sample_prob_df.loc[state_id]
most_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=False).head(5)
least_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=True).head(5)
print("Table cards:", row["table"])
print("Predicted excess rank:", prob_df.loc[state_id, "pred"])
print("Actual excess rank", df.loc[state_id, "excess_rank"])
print("Most and least likely hands:")
for hand, prob in most_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")
print("...")
for hand, prob in least_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")

Table cards: ♦ 9 ♦ J ♦ A 
Predicted excess rank: 2
Actual excess rank 1
Most and least likely hands:
♥ 9 ♠ A : 1.46%
♣ J ♣ A : 1.46%
♥ J ♥ A : 1.46%
♣ 9 ♠ A : 1.46%
♣ J ♠ A : 1.46%
...
♥ J ♠ J : 0.00%
♣ J ♠ J : 0.00%
♥ A ♣ A : 0.00%
♣ A ♠ A : 0.00%
♥ A ♠ A : 0.00%


### Attempt to infer winning probabilities from hidden state probabilities

In [ ]:
hand_winning_probs = []
for i, (state_id, row) in enumerate(hand_probs_df.iterrows()):
    print(f"Processing row {i}", flush=True)
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    processed_row = df.loc[state_id]
    n_players = processed_row["n_players"]
    hand_winning_probs.append(CheatSheet.get_all_winning_probabilities(table_cards, n_players, 1000))

hand_winning_probs_df = pd.DataFrame(hand_winning_probs, index=hand_probs_df.index)
hand_winning_probs_df